In [ ]:
from datetime import date
import netCDF4 as nc
import os
import pandas as pd
import sys
import time
import fiona
import xarray as xr

In [ ]:
sys.path.append('D:\\repos\\E-OBS-SWB2\\Python') #path dove hai i file eobsobject e rechargecalc
from EOBSobject import EOBSobject
from RechargeCalc import RechargeCalc

In [ ]:
cwd = '' #folder principale di swb
cwd

'D:/Dati pesanti/SWB2_MAURICE'

## Use EOBSobject

In [ ]:
outpath = os.path.join(cwd, 'climate_ncfile')
inpath = os.path.join(cwd, 'data_original', 'E-OBS')

In [ ]:
vars = ['rr', 'tn', 'tx']
outnames = ['prcp', 'tmin', 'tmax']

coord = {'lon': [8.691, 8.929, 9.524, 9.537],
            'lat': [45.611, 45.308, 45.610, 45.306]}
coord = pd.DataFrame(coord)
# need one file per year
start = 2019
end = 2024

In [ ]:
for i, var in enumerate(vars):
    f = EOBSobject(var, inpath, outpath, folder = True, swb2 = True)
    f.load()
    # cut in space and time
    f.set_outname(outnames[i])
    f.cut_spacetime(coord, start, end, option = 'singleyear', contourcell=2, autosave = True, readme = True)
    f.close_netcdf()

In [ ]:
# Now, launch SWB2

In [ ]:
start = time.time()

cell_area = 100*100 #m2 # dimensione della cella
#Path to the SWB2 output
#  modificare nome file
#  swb2path = os.path.join(os.path.join(cwd, 'output','ModelMI_net_infiltration__2018-01-01_2019-12-31__338_by_660.nc'))
#Path to the input .csv files folder
# modificare folder
# inputpath = os.path.join(cwd,'data_original', 'file_input_rechargecalc', 'swb_MODELMI19')
# modificare nome file
# sppath = os.path.join(inputpath, 'rirrigua_speciale_swb_MODELMI19.csv')

r = RechargeCalc(cell_area, uniqueid = 'indicatore', nSP = 8)
r.load_inputfiles(swb2path, inputpath)

SP1 = 90   #days, 01/01 - 30/03
SP2 = 76   #days, 01/04 - 12/06
SP3 = 92   #days, 13/06 - 15/09
SP4 = 107  #days, 16/09 - 31/12
SPs = [SP1, SP2, SP3, SP4]

r.meteoricR(SPs, units = 'ms', fixrow=1, fixcol=4)

coeffs = {
    'E': 0.3,  #Irrigation technique efficiency
    'R': 0.05, #Residual runoff
    'RISP': 1, #1 - fraction of water saved by a change of irrigation technique
    'P': 1     #Percentage of the cell covered by the irrigation
    }

col = ['land_cover', 'land_cover', 'zona_urbana']
valcol = [123, 124, 1]
option = [0, 1] #0: OR, 1: AND

r.urbanR(coeff=0.125, col=col, valcol=valcol, option=option)

r.irrigationR(coeffs, specialpath=sppath)

r.totalR(fillna=True)

r.export('recharge','rtot',
            #  outpath = os.path.join(cwd, 'rtot'), serve creare la cartella rtot
             outname = 'rtot_swb_MODELMI19_v2',
             withcoord=True,
             coordpath = os.path.join(inputpath, 'coord.csv'))
r.georef('recharge','rtot',
            # outpath = os.path.join(cwd, 'rtot'), serve creare la cartella rtot
            fname = 'rtot_swb_MODELMI19_v2.shp', 
            coordpath = os.path.join(inputpath, 'coord.csv'),
            crs = 'epsg:3003', dropcoord=False, driver = 'ESRI Shapefile')
end = time.time()
print((end - start)/60, 'min')